# Digit Recognizer (MNIST)

**Competition:** [https://www.kaggle.com/competitions/digit-recognizer](https://www.kaggle.com/competitions/digit-recognizer)

**Task:** Classify 28x28 handwritten digit images (0-9).

**Metric:** Accuracy  &nbsp;|&nbsp;  **Public score achieved:** 0.973

## Approach
- 784 pixel features scaled to [0,1].
- LightGBM multi-class with early stopping on a 10% holdout.

The cell below is a self-contained LightGBM pipeline. It reads the attached
competition data, prints the cross-validation score, and writes `submission.csv`.


In [ ]:
import warnings, numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold, KFold, train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, mean_squared_error
import lightgbm as lgb
warnings.filterwarnings("ignore")

def to_category(train, test, cols):
    for c in cols:
        train[c] = train[c].astype("category"); test[c] = test[c].astype("category")
        cats = pd.api.types.union_categoricals([train[c], test[c]]).categories
        train[c] = pd.Categorical(train[c], categories=cats)
        test[c] = pd.Categorical(test[c], categories=cats)
    return train, test

CLF = dict(n_estimators=2000, learning_rate=0.03, num_leaves=63, subsample=0.8,
           subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0,
           min_child_samples=30, random_state=42, n_jobs=-1, verbose=-1)
REG = dict(n_estimators=3000, learning_rate=0.02, num_leaves=31, subsample=0.8,
           subsample_freq=1, colsample_bytree=0.7, reg_lambda=1.0,
           min_child_samples=20, random_state=42, n_jobs=-1, verbose=-1)
CB = [lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)]

def cv_binary(X, y, Xt, proba=True, n=5):
    oof = np.zeros(len(X)); pred = np.zeros(len(Xt))
    for tr, va in StratifiedKFold(n, shuffle=True, random_state=42).split(X, y):
        m = lgb.LGBMClassifier(**CLF)
        m.fit(X.iloc[tr], y.iloc[tr], eval_set=[(X.iloc[va], y.iloc[va])], callbacks=CB)
        oof[va] = m.predict_proba(X.iloc[va])[:, 1]; pred += m.predict_proba(Xt)[:, 1] / n
    return oof, pred

def cv_multi(X, y, Xt, n=5):
    k = y.nunique(); oof = np.zeros((len(X), k)); pred = np.zeros((len(Xt), k))
    for tr, va in StratifiedKFold(n, shuffle=True, random_state=42).split(X, y):
        m = lgb.LGBMClassifier(**CLF)
        m.fit(X.iloc[tr], y.iloc[tr], eval_set=[(X.iloc[va], y.iloc[va])], callbacks=CB)
        oof[va] = m.predict_proba(X.iloc[va]); pred += m.predict_proba(Xt) / n
    return oof, pred

def cv_reg(X, y, Xt, n=5):
    oof = np.zeros(len(X)); pred = np.zeros(len(Xt))
    for tr, va in KFold(n, shuffle=True, random_state=42).split(X):
        m = lgb.LGBMRegressor(**REG)
        m.fit(X.iloc[tr], y.iloc[tr], eval_set=[(X.iloc[va], y.iloc[va])], callbacks=CB)
        oof[va] = m.predict(X.iloc[va]); pred += m.predict(Xt) / n
    return oof, pred

import glob, os
P = os.path.dirname(sorted(glob.glob("/kaggle/input/**/train.csv", recursive=True))[0])
print("DATA DIR:", P)
tr = pd.read_csv(P + "/train.csv"); te = pd.read_csv(P + "/test.csv")
y = tr["label"]; X = tr.drop(columns=["label"]) / 255.0; Xt = te / 255.0
Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=0.1, stratify=y, random_state=42)
params = dict(n_estimators=1500, learning_rate=0.05, num_leaves=127, subsample=0.8,
              subsample_freq=1, colsample_bytree=0.6, reg_lambda=1.0, random_state=42, n_jobs=-1, verbose=-1)
m = lgb.LGBMClassifier(**params)
m.fit(Xtr, ytr, eval_set=[(Xva, yva)], callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
print("Holdout accuracy:", accuracy_score(yva, m.predict(Xva)))
pred = m.predict(Xt)
pd.DataFrame({"ImageId": np.arange(1, len(Xt) + 1), "Label": pred}).to_csv("submission.csv", index=False)
